In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

In [ ]:
# Taxi-Datensatz von https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# Data Dictionary: https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf
df = pd.read_parquet('daten/yellow_tripdata_2024-04.parquet')

In [ ]:
# Überblick
df.describe().round(3)

In [ ]:
# Features (=neue Variablen bzw. Werte), die wir aus bestehenden ableiten
df["duration"]     = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
df["duration_min"] = df["duration"].dt.total_seconds() / 60
df["avg_speed"]    = df["trip_distance"] / (df['duration_min'] / 60)
df["tip_pct"]      = df["tip_amount"] / df["total_amount"]

In [ ]:
# Wir erzeugen einen neuen Dataframe, bei dem wir unerwünschte Werte ausfiltern
# Wir haben die Grenzen der Datenfilterung willkürlich gewählt: auf Basis unseres Vorwissens
# bzw. unserer Fragestellung. Es gibt selten eindeutige Kriterien dafür.
df2 = df.query((
    "total_amount > 1 and "
    "tolls_amount < 100 and "
    "trip_distance > 0.1 and trip_distance < 100 and "
    "duration_min > 1 and duration_min < 180 and "
    "avg_speed > 1 and avg_speed < 75 and "
    'tpep_pickup_datetime > "2024-04-01 00:00:00" and '
    "tpep_dropoff_datetime > tpep_pickup_datetime"
))

In [ ]:
df2.shape
# Wir haben inzwischen über 100.000 Datensätze herausgefiltert. Das entspricht ca. 3% der Datensätze.

In [ ]:
# Da folgende Berechnungen teils recht lange dauern können,
# nehmen wir nur ein Sample des gesamten Datensatzes
df3 = df2.sample(10000, random_state=1234)

In [ ]:
# Wir stellen ein Modell auf, bei der wir Entfernung, Dauer und Trinkgeld mit einfließen lassen.
model = smf.ols("total_amount ~ trip_distance + duration_min + tip_amount",
                data=df3).fit()

In [ ]:
model.summary()

In [ ]:
# Residuen als Scatterplot in Abhängigkeit vom verhergesagten Wert
sns.scatterplot(x=model.fittedvalues, y=model.resid, alpha=0.1)

In [ ]:
# y vs ŷ Plot (echte vs. vorhergesagte Daten)
sns.scatterplot(x=df3["total_amount"], y=model.fittedvalues, alpha=0.1)